# Lab: Motifs and Modules in Biological Networks

---

This lab is a continuation of the previous session on the yeast (*S. cerevisiae*) PPI network.
We now move from global network properties to **local patterns (motifs)** and **functional organization (modules)**.


### Library availability

| Algorithm | Available in networkx? | Package used here |
|---|---|---|
| Triadic census | `nx.triadic_census()` | networkx |
| Louvain | `nx.community.louvain_communities()` | networkx |
| Modularity Q | `nx.community.modularity()` | networkx |
| Leiden | not in networkx | `cdlib` |
| MCL | not in networkx | `cdlib` |
| MCODE | not in networkx | `cdlib` |

`CDlib` is a meta-library for community detection in complex networks. 

---

## 0. Install dependencies

In [ ]:
!pip install cdlib leidenalg igraph

## 1. Setup

In [ ]:
import requests
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
from io import StringIO
from cdlib import algorithms
import os, json

np.random.seed(1024)
print(f'networkx version: {nx.__version__}')

### 1.1 Reload the yeast PPI network (undirected graph)

In [ ]:
CACHE_FILE = 'string_yeast_ppi.csv'

print('Loading cached network...')
df_raw = pd.read_csv(CACHE_FILE)

G_real = nx.from_pandas_edgelist(df_raw,'preferredName_A','preferredName_B',edge_attr='score')

N, E = G_real.number_of_nodes(), G_real.number_of_edges()
print(f'Working network: N={N}, E={E}, <k>={2*E/N:.2f}')

### 1.2 Load the yeast directed function network (directed graph)

**Note:** STRING functional links are used here as a proxy for a directed network. This is **not** a curated transcription regulatory network, so edge directions do not necessarily represent transcriptional regulation.

In [ ]:
SPECIES    = 4932   # S. cerevisiae
SCORE_THR  = 700    # high confidence (0-1000)
TRN_CACHE = 'string_yeast_trn.tsv'
TRN_SEED_N = 100

if os.path.exists(TRN_CACHE):
    df_trn = pd.read_csv(TRN_CACHE, sep='\t')
else:
    top_nodes = [n for n, d in sorted(G_real.degree(), key=lambda x: x[1], reverse=True)[:TRN_SEED_N]]
    r = requests.get('https://string-db.org/api/tsv/network',
                     params={'identifiers':'%0d'.join(top_nodes),
                             'species':SPECIES,'required_score':SCORE_THR,'network_type':'functional','limit':0})
    df_trn = pd.read_csv(StringIO(r.text), sep='\t')
    df_trn.to_csv(TRN_CACHE, sep='\t', index=False)

G_dir = nx.from_pandas_edgelist(df_trn, 'preferredName_A', 'preferredName_B',
                                 edge_attr='score', create_using=nx.DiGraph())
lcc_dir = max(nx.weakly_connected_components(G_dir), key=len)
G_dir   = G_dir.subgraph(lcc_dir).copy()
print(f'Directed network: N={G_dir.number_of_nodes()}, E={G_dir.number_of_edges()}')

---
## 2. Motif detection

**`nx.triadic_census()` is available in networkx** and counts all 16 directed 3-node subgraph types.

MAN notation: digits = number of mutual / asymmetric / null edges.

Key types: 
  - **030T** = feed-forward loop, 
  - **030C** = feedback loop, 
  - **021D** = fan-out.

All types: 
  - https://networkx.org/documentation/stable/reference/algorithms/triads.html
  - https://networkx.org/documentation/stable/auto_examples/graph/plot_triad_types.html

### 2.1 Three-node subgraph census

> comment: most common is empty type 003 (it is sparse network)

In [ ]:
census_real = nx.triadic_census(G_dir)

df_census = (pd.DataFrame(list(census_real.items()), columns=['Triad type','Count'])
               .sort_values('Count', ascending=False).reset_index(drop=True))
print('Triadic census:')
print(df_census.to_string(index=False))

### 2.2 Degree-preserving randomization

`nx.double_edge_swap()` is undirected only. We implement directed swap manually.

In [ ]:
def randomize_directed(G, n_swaps_per_edge=10, n_tries_multiplier=100):
    '''Function for swapping edges'''
    H = G.copy(); edges = list(H.edges())
    n_swaps = n_swaps_per_edge * len(edges)
    max_tries = n_tries_multiplier * n_swaps
    swaps_done = 0
    for _ in range(max_tries):
        if swaps_done >= n_swaps: break
        idx = np.random.choice(len(edges), 2, replace=False)
        a,b = edges[idx[0]]; c,d = edges[idx[1]]
        if a==c or b==d or a==d or c==b: continue
        if H.has_edge(a,d) or H.has_edge(c,b): continue
        H.remove_edge(a,b); H.remove_edge(c,d)
        H.add_edge(a,d);    H.add_edge(c,b)
        edges[idx[0]]=(a,d); edges[idx[1]]=(c,b)
        swaps_done += 1
    return H

# here is part for ? 
N_RAND = 100
print(f'Generating {N_RAND} degree-preserving randomizations...')
rand_counts = defaultdict(list)
for _ in range(N_RAND):
    for triad, count in nx.triadic_census(randomize_directed(G_dir)).items():
        rand_counts[triad].append(count)
print(f'Done')

### 2.3 Z-scores and motif identification

$$Z = \frac{N_{\text{real}} - \langle N_{\text{rand}} \rangle}{\sigma_{\text{rand}}}$$

$Z > 2$ $\rightarrow$ motif

$Z < -2$ $\rightarrow$ anti-motif.

In [ ]:
# generate statistics part 
rows = []
for triad, count_real in census_real.items():
    rand = rand_counts[triad]
    mu, sigma = np.mean(rand), np.std(rand)
    z = (count_real - mu) / sigma if sigma > 0 else 0.0
    rows.append({'Triad':triad,'Real':count_real,
                 'Mean rand':round(mu,1),'Std rand':round(sigma,1),'Z-score':round(z,2)})

df_motifs = (pd.DataFrame(rows).sort_values('Z-score', ascending=False)
               .query('Real > 0').reset_index(drop=True))
print(df_motifs.to_string(index=False))

### 2.4 Feed-forward loop (030T) instances

In [ ]:
def find_ffls(G):
    ffls = []
    for X in G.nodes():
        tX = set(G.successors(X))
        for Y in tX:
            for Z in tX & set(G.successors(Y)):
                if Z != Y: ffls.append((X,Y,Z))
    return ffls

ffls = find_ffls(G_dir)
print(f'Found {len(ffls)} feed-forward loop instances.')
print('\nTop 10 "functional" regulators:')
for p, n in Counter(f[0] for f in ffls).most_common(10):
    print(f'  {p}: {n} FFLs')

**Warning:** `G_dir` is built from STRING **functional links**, not a curated transcription regulatory network. The "regulators" identified here reflect functional connectivity, not necessarily transcriptional control — proteins such as DNA repair hubs may rank highly without being true transcription factors. For a proper analysis of master regulators use a curated TF-target database such as [YEASTRACT](http://www.yeastract.com).

---

## 3. Module detection

All community detection methods implemented in `cdlib`: https://cdlib.readthedocs.io/en/latest/reference/cd_algorithms/node_clustering.html

**Louvain** is in `networkx` (`nx.community.louvain_communities()`). Here, we use Leiden instead — it fixes the resolution limit and guarantees well-connected communities. 

Modularity $Q$ computation (`nx.community.modularity()`) **is** in `networkx` and works with any partition. *Reminder*: Modularity $Q$ measures the fraction of edges that fall within communities minus the fraction expected in a random network.

### 3.1 Leiden

Not in `networkx`; use `cdlib.algorithms.leiden()`

In [ ]:
leiden_result = algorithms.leiden(G_real, weights='score')
comms_leiden  = sorted([set(c) for c in leiden_result.communities], key=len, reverse=True)
Q_leiden      = nx.community.modularity(G_real, comms_leiden)  # Q computed via networkx

print(f'Leiden: {len(comms_leiden)} communities, Q = {Q_leiden:.4f}')
for i, c in enumerate(comms_leiden[:10]):
    print(f'  Module {i+1}: {len(c)} nodes')

# For reference: Louvain IS available in networkx
comms_louvain = nx.community.louvain_communities(G_real, seed=42)
Q_louvain = nx.community.modularity(G_real, comms_louvain)
print(f'\n[Reference] Louvain (networkx): {len(comms_louvain)} communities, Q = {Q_louvain:.4f}')

### 3.2 MCL (Markov Clustering)

Not in `networkx`; use `cdlib.algorithms.markov_clustering()`

In [ ]:
mcl_result = algorithms.markov_clustering(G_real, inflation=2.0)
comms_mcl  = sorted([set(c) for c in mcl_result.communities if len(c) > 1], key=len, reverse=True)
Q_mcl      = nx.community.modularity(G_real, comms_mcl)

print(f'MCL: {len(comms_mcl)} communities (size>1), Q = {Q_mcl:.4f}')
for i, c in enumerate(comms_mcl[:10]):
    print(f'  Module {i+1}: {len(c)} nodes')

### 3.3 MCODE

Not in `networkx`; use `cdlib.algorithms.mcode()`

MCODE is **local** and produces **overlapping** modules unlike Leiden and MCL.

In [ ]:
mcode_result = algorithms.mcode(G_real)
comms_mcode  = sorted([set(c) for c in mcode_result.communities if len(c) > 1], key=len, reverse=True)

print(f'MCODE: {len(comms_mcode)} modules (size>1)')
for i, c in enumerate(comms_mcode[:10]):
    print(f'  Module {i+1}: {len(c)} nodes')

# Check for overlapping nodes
all_nodes = [node for c in comms_mcode for node in c]
overlap   = {n: count for n, count in Counter(all_nodes).items() if count > 1}
print(f'\nProteins in >1 module: {len(overlap)} (e.g. {list(overlap.keys())[:5]})')

### 3.4 Visualization

In [ ]:
def assign_colors(G, communities, cmap=plt.cm.tab20):
    nc = {}
    for i, c in enumerate(communities):
        col = cmap(i / max(len(communities), 1))
        for n in c: nc[n] = col
    return [nc.get(n, (0.8,0.8,0.8,1.0)) for n in G.nodes()]

fig, axes = plt.subplots(1, 3, figsize=(20, 7))
pos     = nx.spring_layout(G_real, seed=1024, k=0.6) #in new version use: method='force'
degrees = dict(G_real.degree())

for ax, comms, title in zip(axes,
        [comms_leiden, comms_mcl, comms_mcode],
        ['Leiden', 'MCL', 'MCODE']):
    nx.draw_networkx_edges(G_real, pos, ax=ax, alpha=0.15, edge_color='gray', width=0.5)
    nx.draw_networkx_nodes(G_real, pos, ax=ax,
                           node_color=assign_colors(G_real, comms),
                           node_size=[10+3*degrees[n] for n in G_real.nodes()], alpha=0.85)
    ax.set_title(f'{title}\n({len(comms)} modules)', fontsize=12, fontweight='bold')
    ax.axis('off')

plt.suptitle('All modules colored', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 3.5 Module statistics

In [ ]:
def module_stats(G, communities, label, top_n=10):
    rows = []
    for i, c in enumerate(communities[:top_n]):
        sg = G.subgraph(c); n, e = sg.number_of_nodes(), sg.number_of_edges()
        mx = n*(n-1)/2
        rows.append({'Module':i+1,'Size':n,'Edges':e,
                     'Density':round(e/mx,3) if mx>0 else 0,
                     '<C>':round(nx.average_clustering(sg),3)})
    print(f'\n=== {label} ==='); print(pd.DataFrame(rows).to_string(index=False))

module_stats(G_real, comms_leiden, 'Leiden')
module_stats(G_real, comms_mcl,    'MCL')
module_stats(G_real, comms_mcode,  'MCODE')

---

## 4. Exercises

### Exercise 1: Effect of null model on motif detection

Repeat the Z-score analysis using an **Erdős–Rényi graph** as the null model.

Use `nx.gnm_random_graph(N, E, directed=True)` to match both $N$ and $E$.

**Questions:**
1. Which triad types are motifs with the ER null model? Are they the same as with the degree-preserving model?
2. Which null model is more appropriate for biological motif analysis, and why?

In [ ]:
# params 
## network params 
N = G_dir.number_of_nodes()
E = G_dir.number_of_edges()
## experiments params
N_RAND = 100 

# analysis
## generating random networks 
print(f'Generating {N_RAND} Erdős-Rényi random networks...')
er_counts = defaultdict(list)
## use index for seed seetings (reproducible results)
for i in range(N_RAND):
    ### Create random directed graph with same params as original network 
    G_er = nx.gnm_random_graph(N, E, seed=i, directed=True)
    census_er = nx.triadic_census(G_er)
    
    for triad, count in census_er.items():
        er_counts[triad].append(count)

print('Done generating. Calculating statistics...')

## compare results 
### initialization of table 
rows = []
census_real = nx.triadic_census(G_dir) # Ensure we have the results for the real network

### calculating differences between networks 
for triad, count_real in census_real.items():
    rand_data = er_counts[triad]
    mu = np.mean(rand_data)
    sigma = np.std(rand_data)
    
    # Z-score: measures how many standard deviations the real count is from the random mean
    z = (count_real - mu) / sigma if sigma > 0 else 0.0
    
    rows.append({
        'Triad': triad,
        'Real': count_real,
        'Mean_ER': round(mu, 1),
        'Std_ER': round(sigma, 1),
        'Z-score': round(z, 2)
    })

# display results 
df_motifs_er = (pd.DataFrame(rows)
                .sort_values('Z-score', ascending=False)
                .query('Real > 0')
                .reset_index(drop=True))

## display 
print(df_motifs_er.to_string(index=False)) # for keeping resulst
# df_motifs_er

### Exercise 2: MCL inflation parameter

Run MCL with $r \in \{1.5, 2.0, 2.5, 3.0, 3.5, 4.0\}$. Plot number of modules and average size vs $r$.

MCL: `algorithms.markov_clustering(G_real, inflation=r)` (cdlib)

**Questions:**
1. What happens to the number and size of modules as inflation $r$ increases?
2. From the plot of the number of modules as a function of $r$: Can you identify an **elbow point**? What value of $r$ does it suggest for this network?

In [ ]:
r_params = [1.5, 2.0, 2.5, 3.0, 3.5, 4.0]

for r in r_params:
    algorithms.markov_clustering(G_real)

In [ ]:
# 1. Define the range of inflation parameters (r) as specified in the exercise
inflation_values = [1.5, 2.0, 2.5, 3.0, 3.5, 4.0]

# Lists to store results for plotting
num_modules = []
avg_sizes = []
mcl_stats = []

print("Running MCL inflation sweep...")

for r in inflation_values:
    # 2. Run Markov Clustering (MCL) with the current inflation value
    # G_real is our yeast PPI network
    mcl_res = algorithms.markov_clustering(G_real, inflation=r)
    
    # Filter communities to keep only those with more than 1 node (functional modules)
    comms = [c for c in mcl_res.communities if len(c) > 1]
    
    # 3. Calculate metrics
    n_mods = len(comms)
    avg_sz = np.mean([len(c) for c in comms]) if comms else 0
    
    num_modules.append(n_mods)
    avg_sizes.append(avg_sz)
    
    mcl_stats.append({
        'Inflation (r)': r,
        'Num_Modules': n_mods,
        'Avg_Size': round(avg_sz, 2)
    })
    print(f"  r={r}: Found {n_mods} modules, average size {avg_sz:.2f}")

# 4. Visualization of the results
fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot Number of Modules
color = 'tab:blue'
ax1.set_xlabel('Inflation (r)')
ax1.set_ylabel('Number of Modules (size > 1)', color=color)
ax1.plot(inflation_values, num_modules, marker='o', color=color, linewidth=2, label='Number of Modules')
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(True, linestyle='--', alpha=0.6)

# Create a second y-axis for the Average Size
ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Average Module Size', color=color)
ax2.plot(inflation_values, avg_sizes, marker='s', color=color, linewidth=2, label='Avg Size')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('MCL Sensitivity Analysis: Number of Modules and Avg Size vs Inflation (r)')
fig.tight_layout()
plt.show()

# Display summary table
df_mcl_sweep = pd.DataFrame(mcl_stats)
print("\n=== MCL Inflation Sweep Results ===")
print(df_mcl_sweep.to_string(index=False))